# Module 05 — Persistent Memory with AWS AgentCore

In the previous module, our `ConversationSaverHook` stored turns in a Python list — which disappears when the process ends. Real agents need **persistent memory** that survives across sessions.

**AWS AgentCore Memory** provides two tiers:

| Tier | What it stores | How it works |
|---|---|---|
| Short-term | Raw conversation events | You write them explicitly via `create_event` |
| Long-term | Extracted facts & preferences | AgentCore processes events in the background and extracts structured knowledge |

The magic: you save raw conversations, and AgentCore automatically extracts things like *"user is vegetarian"* or *"user loves spicy food"* into a queryable knowledge store.

---

### Architecture

```
User message
    ↓
Agent responds
    ↓
AfterInvocationHook → create_event() → Short-term memory
                                              ↓ (background, async)
                                       Long-term memory (extracted preferences)
                                              ↓
AgentInitializedHook → retrieve_memories() → injected into system prompt
```

In [ ]:
import os
import logging
from datetime import datetime
from botocore.exceptions import ClientError
from bedrock_agentcore.memory import MemoryClient
from bedrock_agentcore.memory.constants import StrategyType

logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")
logger = logging.getLogger("memory-demo")

REGION = os.getenv('AWS_REGION', 'us-east-1')
USER_ID = "workshop-user-001"
SESSION_ID = f"session_{datetime.now().strftime('%Y%m%d%H%M%S')}"

print(f"Region     : {REGION}")
print(f"User ID    : {USER_ID}")
print(f"Session ID : {SESSION_ID}")

### Step 1 — Create a Memory resource

A Memory resource is a named container in AgentCore. You create it once and reuse it across sessions.

The **strategy** tells AgentCore what kind of information to extract from conversations.

In [ ]:
client = MemoryClient(region_name=REGION)

# Define what AgentCore should extract from conversations
strategies = [
    {
        StrategyType.USER_PREFERENCE.value: {
            "name": "UserPreferences",
            "description": "Captures user preferences including food, hobbies, and personal details",
            "namespaces": ["user/{actorId}/preferences"]
        }
    }
]

memory_id = None
memory_name = "WorkshopMemory"

try:
    memory = client.create_memory_and_wait(
        name=memory_name,
        strategies=strategies,
        description="Workshop demo memory",
        event_expiry_days=7,
        max_wait=300,
        poll_interval=10
    )
    memory_id = memory['id']
    logger.info(f"Created memory: {memory_id}")

except ClientError as e:
    if "already exists" in str(e):
        memories = client.list_memories()
        memory_id = next((m['id'] for m in memories if m['id'].startswith(memory_name)), None)
        logger.info(f"Using existing memory: {memory_id}")
    else:
        raise

print(f"\nMemory ID: {memory_id}")
print("Save this for the next session!")

### Step 2 — Save a conversation event (short-term memory)

In [ ]:
# Simulate saving a conversation turn
event = client.create_event(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id=SESSION_ID,
    messages=[
        ("I love spicy food, especially Thai and Indian cuisines!", "USER"),
        ("Great! I'll keep that in mind for future recommendations.", "ASSISTANT")
    ]
)
print("Saved event:", event)

In [ ]:
# Save another turn
client.create_event(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id=SESSION_ID,
    messages=[
        ("By the way, I'm vegetarian and allergic to peanuts.", "USER"),
        ("Noted! I'll always suggest vegetarian, peanut-free options.", "ASSISTANT")
    ]
)
print("Saved second event")

### Step 3 — Read short-term memory

In [ ]:
events = client.list_events(
    memory_id=memory_id,
    actor_id=USER_ID,
    session_id=SESSION_ID
)

print(f"Found {len(events)} events in short-term memory:")
for i, event in enumerate(events, 1):
    print(f"\nEvent {i}:")
    for item in event.get('payload', []):
        conv = item.get('conversational', {})
        role = conv.get('role', '')
        text = conv.get('content', {}).get('text', '')
        print(f"  [{role}]: {text}")

### Step 4 — Query long-term memory (extracted preferences)

AgentCore processes events in the background and extracts structured preferences. This may take 30-60 seconds after saving events.

In [ ]:
import time

print("Waiting 30 seconds for background processing...")
time.sleep(30)

preferences = client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"user/{USER_ID}/preferences",
    query="food preferences dietary restrictions allergies",
    top_k=5
)

print(f"\nExtracted {len(preferences)} long-term preferences:")
for i, pref in enumerate(preferences, 1):
    content = pref.get('content', {})
    text = content.get('text', '') if isinstance(content, dict) else ''
    print(f"{i}. {text}")

### Step 5 — Simulate a new session using retrieved memory

This is the key pattern: start a new session, retrieve long-term memory, inject it into the system prompt.

In [ ]:
from strands import Agent

# Retrieve preferences for the new session
preferences = client.retrieve_memories(
    memory_id=memory_id,
    namespace=f"user/{USER_ID}/preferences",
    query="food preferences dietary restrictions",
    top_k=5
)

# Format them for the system prompt
pref_lines = []
for pref in preferences:
    content = pref.get('content', {})
    text = content.get('text', '') if isinstance(content, dict) else ''
    if text:
        pref_lines.append(f"- {text}")

system_prompt = "You are a helpful food assistant. Be concise."
if pref_lines:
    system_prompt += "\n\n## What I know about this user:\n" + "\n".join(pref_lines)

print("System prompt for new session:")
print(system_prompt)
print()

# New session — agent has no conversation history, but has the user's preferences
new_session_agent = Agent(
    model="us.anthropic.claude-3-5-haiku-20241022-v1:0",
    system_prompt=system_prompt
)

new_session_agent("Hi! I'm hungry. What should I eat tonight?")

The agent remembered the user's preferences across sessions — without the user having to repeat themselves.

---

### Key takeaways

- `MemoryClient` is the entry point for AgentCore memory
- `create_memory_and_wait` creates a named memory resource with extraction strategies
- `create_event` saves raw conversation turns (short-term memory)
- `retrieve_memories` queries extracted preferences (long-term memory)
- The extraction happens asynchronously in the background
- Inject retrieved preferences into the system prompt at session start

Next up: **06_full_agent.ipynb** — wiring everything together with Bedrock Guardrails.